In [ ]:
# %% [markdown]
# # Setup
# Read all CSVs in the current directory and prepare for plotting

import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

# SciPy KDE for smooth distributions
from scipy.stats import gaussian_kde

# Find all CSV files in the current working directory
csv_files = sorted(glob.glob("*.csv"))

print("Found CSV files:")
for f in csv_files:
    print("  -", f)

In [ ]:
# %% 
# Helper: smooth KDE-based distribution curves with filled area and mean/SD in legend

def plot_kde_distributions(data_dict, xlabel, title, num_points=512, bw_method=None):
    """
    data_dict: dict {label: 1D array-like}
    Plots smooth KDE curves (Gaussian kernel) for all datasets, overlapping,
    with area under the curve filled and legend showing mean and SD.
    """
    # Collect all values to define global x-range
    all_values_list = []
    for v in data_dict.values():
        v = np.asarray(v)
        v = v[~np.isnan(v)]
        if v.size > 0:
            all_values_list.append(v)

    if not all_values_list:
        print("No data to plot for:", title)
        return

    all_values = np.concatenate(all_values_list)
    data_min, data_max = np.min(all_values), np.max(all_values)

    if data_max == data_min:
        # Degenerate case: all values identical
        data_min -= 1.0
        data_max += 1.0

    # Add a bit of padding and build common x-grid
    padding = 0.02 * (data_max - data_min)
    x_grid = np.linspace(data_min - padding, data_max + padding, num_points)

    plt.figure(figsize=(8, 5))

    for label, values in data_dict.items():
        values = np.asarray(values)
        values = values[~np.isnan(values)]
        if values.size == 0:
            continue

        mean_val = np.mean(values)
        std_val = np.std(values, ddof=1) if values.size > 1 else 0.0

        # Gaussian KDE
        if bw_method is not None:
            kde = gaussian_kde(values, bw_method=bw_method)
        else:
            kde = gaussian_kde(values)  # default bandwidth (Scott's rule)

        y = kde(x_grid)

        # Legend label: name + mean and SD
        legend_label = f"{label} (mean={mean_val:.2f}, sd={std_val:.2f})"

        # Plot line
        plt.plot(x_grid, y, alpha=0.7, label=legend_label)

        # Fill under curve
        plt.fill_between(x_grid, y, alpha=0.3)

    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# %% 
# Twist KDE curves (2nd column of each CSV)

twist_data = {}

for f in csv_files:
    df = pd.read_csv(f)

    # Expect format: timestep, twist, bend
    if df.shape[1] < 2:
        continue

    twist = pd.to_numeric(df.iloc[:, 1], errors="coerce")  # 2nd column
    label = os.path.splitext(os.path.basename(f))[0]        # filename without .csv
    twist_data[label] = twist.values

plot_kde_distributions(
    twist_data,
    xlabel="Twist (deg)",
    title="Twist distributions curves",
    # bw_method can be tuned if needed, e.g. 0.8, 1.2, "silverman"
    bw_method=0.2
)

In [ ]:
# %% 
# Bend KDE curves (3rd column of each CSV)

bend_data = {}

for f in csv_files:
    df = pd.read_csv(f)

    if df.shape[1] < 3:
        continue

    bend = pd.to_numeric(df.iloc[:, 2], errors="coerce")   # 3rd column
    label = os.path.splitext(os.path.basename(f))[0]       # filename without .csv
    bend_data[label] = bend.values

plot_kde_distributions(
    bend_data,
    xlabel="Bend (deg)",
    title="Bend distributions curves",
    bw_method=0.2
)